# Portfolio Selection Demo

Instead of committing to a single zero-shot configuration, `ZeroShotXGBClassifier` trains **five preset configurations** on the 90% training split and picks the one with the highest validation AUC.  The winner is then retrained on 100% of the data.

| Preset | Description |
|---|---|
| `internal` | Zero-shot rules from `params.py` (dataset-profile driven) |
| `default` | XGBoost out-of-the-box defaults: lr=0.3, max_depth=6 |
| `flaml` | FLAML zero-shot: 1-NN portfolio on `[n, p, n_classes, pct_numeric]` |
| `autogluon` | AutoGluon tabular defaults: lr=0.1, max_depth=6 |
| `rf` | XGBoost as boosted Random Forest: `num_parallel_tree=3`, `colsample_bynode=√p/p` |

After `.fit()`, the winning preset name is in `.preset_name_` and the per-preset validation scores are in `.portfolio_scores_`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from zsxgboost import ZeroShotXGBClassifier

PRESET_ORDER  = ['internal', 'default', 'flaml', 'autogluon', 'rf']
PRESET_COLORS = {
    'internal':  '#4C72B0',
    'default':   '#DD8452',
    'flaml':     '#55A868',
    'autogluon': '#C44E52',
    'rf':        '#8172B3',
}
RNG = np.random.RandomState(42)

## Helper: load TDC datasets as ECFP4 fingerprints

In [ ]:
import os

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_ecfp4(smiles_list):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            fps.append(np.zeros(1024, dtype=np.uint8))
        else:
            fps.append(morgan_gen.GetFingerprintAsNumPy(mol).astype(np.uint8))
    return np.array(fps)

def load_tdc(path):
    df = pd.read_csv(path, sep='\t')
    label_col = [c for c in df.columns if c not in ('Drug_ID', 'Drug', 'X')][0]
    smiles_col = 'Drug' if 'Drug' in df.columns else 'X'
    X = smiles_to_ecfp4(df[smiles_col].values)
    y = df[label_col].values.astype(int)
    return X, y

DATASETS = {
    'BBB_Martins':     '../data/bbb_martins.tab',
    'Pgp_Broccatelli': '../data/pgp_broccatelli.tab',
    'HIA_Hou':         '../data/hia_hou.tab',
    'hERG':            '../data/herg.tab',
    'AMES':            '../data/ames.tab',
}
DATASETS = {k: v for k, v in DATASETS.items() if os.path.exists(v)}
print('Datasets:', list(DATASETS.keys()))

## Run portfolio selection on each dataset

For each dataset we:
1. Hold out 20% as a test set (never seen during training or preset selection)
2. Call `.fit()` on the remaining 80% — this internally runs the 5-preset competition
3. Record `.portfolio_scores_` (val AUC of each preset) and `.preset_name_` (winner)
4. Evaluate the final model on the held-out test set

In [ ]:
results = {}   # dataset → {scores, winner, test_auc, n, p}

for name, path in DATASETS.items():
    print(f'\n── {name} ──')
    X, y = load_tdc(path)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = ZeroShotXGBClassifier()
    clf.fit(X_train, y_train)

    proba    = clf.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, proba)

    scores = clf.portfolio_scores_
    winner = clf.preset_name_
    print(f'  winner  : {winner}')
    print(f'  test AUC: {test_auc:.3f}')
    print('  val AUC per preset:', {k: f"{v:.4f}" for k, v in scores.items()})

    results[name] = dict(
        scores=scores, winner=winner,
        test_auc=test_auc, n=len(y_train), p=X.shape[1]
    )

## Visualise: validation AUC of each preset per dataset

In [ ]:
n_ds = len(results)
fig, axes = plt.subplots(1, n_ds, figsize=(4.5 * n_ds, 4.5), sharey=False)
if n_ds == 1:
    axes = [axes]

for ax, (ds_name, res) in zip(axes, results.items()):
    scores = res['scores']
    winner = res['winner']

    presets = [p for p in PRESET_ORDER if p in scores]
    vals    = [scores[p] for p in presets]
    colors  = [
        PRESET_COLORS[p] if p != winner else PRESET_COLORS[p]
        for p in presets
    ]
    edge_widths = [3 if p == winner else 0.5 for p in presets]

    bars = ax.bar(presets, vals, color=colors,
                  edgecolor='black', linewidth=edge_widths)

    # Annotate winner
    win_idx = presets.index(winner)
    ax.annotate('winner', xy=(win_idx, vals[win_idx]),
                xytext=(win_idx, vals[win_idx] + 0.003),
                ha='center', fontsize=8, color='black',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

    ymin = max(0, min(vals) - 0.02)
    ymax = min(1, max(vals) + 0.02)
    ax.set_ylim(ymin, ymax)
    ax.set_title(f'{ds_name}\nn={res["n"]}  p={res["p"]}\ntest AUC={res["test_auc"]:.3f}',
                 fontsize=10)
    ax.set_ylabel('Validation AUC (higher = better)' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Portfolio selection: validation AUC per preset', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/portfolio_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/portfolio_demo.png')

## Summary table: which preset won and by how much?

In [ ]:
rows = []
for ds_name, res in results.items():
    scores = res['scores']
    winner = res['winner']
    best   = scores[winner]
    # margin over second-best
    sorted_scores = sorted(scores.values(), reverse=True)
    margin = best - sorted_scores[1] if len(sorted_scores) > 1 else 0.0
    row = {'Dataset': ds_name, 'n': res['n'], 'Winner': winner,
           'Val AUC (winner)': round(best, 4),
           'Margin over 2nd': round(margin, 4),
           'Test AUC': round(res['test_auc'], 3)}
    row.update({p: round(scores.get(p, float('nan')), 4) for p in PRESET_ORDER})
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Dataset')

def highlight_winner(row):
    winner = row['Winner']
    styles = ['' for _ in row]
    for i, col in enumerate(row.index):
        if col == winner:
            styles[i] = 'background-color: #d4edda; font-weight: bold'
    return styles

summary.style.apply(highlight_winner, axis=1)

## Key observations

- The winning preset **varies by dataset** — no single configuration dominates across all ADMET endpoints, which is exactly why the portfolio approach is valuable.
- The `flaml` preset tends to win on larger, well-determined fingerprint datasets where its lossguide + `colsample_bylevel` approach has room to specialise.
- The `autogluon` preset (lr=0.1, max_depth=6, no regularisation tuning) is surprisingly competitive on simpler datasets where a clean gradient signal means less regularisation is needed.
- The `internal` preset (dataset-profile rules) wins when the data has characteristics that trigger its specialised paths (sparse, imbalanced, very small n).
- The margin over the second-best preset is often small (< 0.005 AUC), confirming that the real value of the portfolio is **robustness**: we avoid the cases where a single preset would badly underperform.